# Validate a MOC sequence, then ingest the final one

The MOC turns our calendar into a command sequence (`S<yy>W<ww>.seq.json`), trimming science around ground contacts and maintenance. The weekly loop:

1. **Validate** each draft: `validate_sequence` compares it against the calendar and writes `<seq_id>_validation_report.txt` beside the file, printing the summary. It writes **nothing** to the database. High-priority truncations go back to the MOC.
2. **Ingest** the final sequence: `ingest_sequence` records it and stamps every observation `SCHEDULED`, `TRUNCATED`, or `DROPPED`.

Two optional inputs sharpen the checks: `ksat_contacts.json` (which contact cut an observation) and the telecom sequence `T<yy>W<ww>.seq.json` (no command collisions with science; telecom activity correlates with contacts).

In [1]:
from pathlib import Path

from pandoraobservations.sequences import validate_sequence

repo = Path.cwd().parent
examples = repo / "examples"
calendar_xml = examples / "PAN-SCICAL-SCI-20260819-VF-20260824-EX-20260831-R002.xml"

result = validate_sequence(
    examples / "S26W35.seq.json",
    calendar_xml,
    contacts_path=examples / "ksat_contacts.json",
    telecom_path=examples / "T26W35.seq.json",
)

2026-08-21 16:22:30 WARNING: PAN-SCICAL-SCI-20260819-VF-20260824-EX-20260831-R002.xml claims 26 visits / 227       
sequences but contains 25 / 226; recording both.

# Calendar vs Sequence Comparison Report

Sequence: S26W35 vs calendar PAN-SCICAL-SCI-20260819-VF-20260824-EX-20260831 R002
Requested observations (all priorities): 226
Requested observations (priority >= 1): 90
Generated observation blocks: 223
Generated blocks matched to a request: 223
PAYLOAD_READ end buffer: 45 s
Start tolerance: 0 s

# Truncation summary
Dropped observations (priority >= 1): 0 (0.00 min)
Split observations (priority >= 1, >1 generated block): 2
Truncated observations (priority >= 1): 8 of 90
Total minutes truncated: 52.00
Worst truncation: 16.62 min (visit=0002 obs_id=022 target=TOI-181b priority=1)

Per-observation truncation (minutes), priority descending:
  2026-08-24T17:07:00.000 visit=0002 obs_id=022 target=TOI-181b priority=1 truncated=16.62 (start=0.00, gap=0.00, end=16.62)
  2026-08-24T20:21:00.000 visit=0002 obs_id=026 target=TOI-181b priority=1 truncated=10.65 (start=0.00, gap=10.65, end=0.00)
  2026-08-24T21:57:00.000 visit=0002 obs_id=028 target=TOI-18

The structured result mirrors `docs/schemas/sequence-record.md`. The pushback list, most important targets first (priority 2 is highest):

In [2]:
import pandas as pd

problems = pd.DataFrame([o for o in result["observations"] if o["scheduled_status"] != "scheduled"])
problems = problems[["obs_id", "target", "priority", "scheduled_status", "requested_s", "coverage_s", "truncation"]]
problems.sort_values(["priority", "coverage_s"], ascending=[False, True]).head(15)

,obs_id,target,priority,scheduled_status,requested_s,coverage_s,truncation
1,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,TOI-181b,1,truncated,3180.0,2183.0,"{'start_late_s': 0.0, 'mid_gap_s': 0.0, 'end_e..."
19,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,HD_189733b,1,truncated,2520.0,2503.0,"{'start_late_s': 0.0, 'mid_gap_s': 0.0, 'end_e..."
2,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,TOI-181b,1,truncated,3180.0,2541.0,"{'start_late_s': 0.0, 'mid_gap_s': 639.0, 'end..."
18,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,HD_189733b,1,truncated,2580.0,2578.0,"{'start_late_s': 2.0, 'mid_gap_s': 0.0, 'end_e..."
3,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,TOI-181b,1,truncated,3240.0,2661.0,"{'start_late_s': 0.0, 'mid_gap_s': 579.0, 'end..."
5,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,HD_189733b,1,truncated,3360.0,2898.0,"{'start_late_s': 462.0, 'mid_gap_s': 0.0, 'end..."
6,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,HD_189733b,1,truncated,3480.0,3098.0,"{'start_late_s': 382.0, 'mid_gap_s': 0.0, 'end..."
8,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,HD_189733b,1,truncated,3360.0,3318.0,"{'start_late_s': 42.0, 'mid_gap_s': 0.0, 'end_..."
4,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,AO_Cassiopeiae,0,dropped,780.0,0.0,"{'start_late_s': 0.0, 'mid_gap_s': 780.0, 'end..."
9,PAN-SCICAL-SCI-20260819-VF-20260824-EX-2026083...,TRAPPIST-1,0,dropped,1440.0,0.0,"{'start_late_s': 0.0, 'mid_gap_s': 1440.0, 'en..."


In [3]:
# Was the worst cut caused by a ground contact, and which one?
worst = max(result["observations"], key=lambda o: sum(o["truncation"].values()))
worst["target"], worst["truncation"], worst["contact_overlaps"]

('TRAPPIST-1',
 {'start_late_s': 0.0, 'mid_gap_s': 1440.0, 'end_early_s': 0.0},
 [{'ground_station': 'Troll Station',
   'antenna': 'TR19',
   'start_utc': '2026-08-26T18:08:20Z',
   'stop_utc': '2026-08-26T18:17:50Z',
   'overlap_s': 570.0}])

Once the MOC delivers the final sequence, ingest it. This writes the sequence record and updates every observation's `scheduled` block and status in the calendar record.

In [4]:
from pandoraobservations.sequences import ingest_sequence

ingest_sequence(
    examples / "S26W35.seq.json",
    calendar_xml,  # its calendar_id is looked up among the ingested records
    contacts_path=examples / "ksat_contacts.json",
    telecom_path=examples / "T26W35.seq.json",
)

# Calendar vs Sequence Comparison Report

Sequence: S26W35 vs calendar PAN-SCICAL-SCI-20260819-VF-20260824-EX-20260831 R002
Requested observations (all priorities): 226
Requested observations (priority >= 1): 90
Generated observation blocks: 223
Generated blocks matched to a request: 223
PAYLOAD_READ end buffer: 45 s
Start tolerance: 0 s

# Truncation summary
Dropped observations (priority >= 1): 0 (0.00 min)
Split observations (priority >= 1, >1 generated block): 2
Truncated observations (priority >= 1): 8 of 90
Total minutes truncated: 52.00
Worst truncation: 16.62 min (visit=0002 obs_id=022 target=TOI-181b priority=1)

Per-observation truncation (minutes), priority descending:
  2026-08-24T17:07:00.000 visit=0002 obs_id=022 target=TOI-181b priority=1 truncated=16.62 (start=0.00, gap=0.00, end=16.62)
  2026-08-24T20:21:00.000 visit=0002 obs_id=026 target=TOI-181b priority=1 truncated=10.65 (start=0.00, gap=10.65, end=0.00)
  2026-08-24T21:57:00.000 visit=0002 obs_id=028 target=TOI-18

WindowsPath('N:/Joe Documents/Gits/pandora-observations/data/sequences/S26W35.seq.json')

In [5]:
from pandoraobservations.cache import load_observations

observations = load_observations()  # data/ is found by walking up from this notebook's folder
observations["status"].value_counts()

status
SCHEDULED    196
TRUNCATED     23
DROPPED        7
Name: count, dtype: int64